<a href="https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question:** Can a simple model rank pages that have the observed decline label above pages that do not?

I use **Logistic Regression** because the target is yes/no and the model is a reasonable first model that is easier to explain than a large ensemble. I compare its probability ranking with the Week-4 rule on the exact same held-out rows.

The label is `trend_direction == "down"`. The label is derived from the trend fields, so those fields and the direct 30-day trend inputs are excluded from model features. IDs are only for grouping and reviewing examples. I also exclude the current 90-day performance totals from model features because they overlap the period used to define the observed label. The Week-4 baseline still uses its original rule, so the comparison is an operational benchmark, not a perfectly equal-information experiment.

This notebook does **not** claim to forecast a future period or prove that refreshing a page causes improvement.


In [1]:
# Load the dataset and define a conservative, leakage-checked feature set.

from pathlib import Path
import json
import subprocess
import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_STATE = 42
TARGET = "is_declining_label"
DATA_FILE = "content_refresh_anonymized.csv"

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
repo_root = next(
    (p for p in search_roots if (p / "work" / "notebooks").is_dir()),
    None,
)

# In Colab, the notebook file may be open without a local repository checkout.
if repo_root is None:
    repo_root = cwd / "flyrank-ml-internship"
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Chaman4211/flyrank-ml-internship.git", str(repo_root)],
            check=True,
        )

csv_candidates = [
    repo_root / "data" / "raw" / DATA_FILE,
    cwd / "data" / "raw" / DATA_FILE,
    cwd / "flyrank-ml-internship-starter" / "data" / "raw" / DATA_FILE,
]

csv_path = next((p for p in csv_candidates if p.is_file() and p.stat().st_size > 0), None)

# If the user's repo does not include the data file, use the public starter repository.
if csv_path is None:
    starter_dir = cwd / "flyrank-ml-internship-starter"
    if not starter_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Chaman4211/flyrank-ml-internship-starter.git", str(starter_dir)],
            check=True,
        )
    candidate = starter_dir / "data" / "raw" / DATA_FILE
    if candidate.is_file() and candidate.stat().st_size > 0:
        csv_path = candidate

if csv_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. Check that the repo/starter data is available."
    )

df = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded dataset: {len(df):,} rows × {df.shape[1]} columns")
print("Data source: anonymized starter CSV; no client names, domains, or URLs are displayed.")

required = {
    "content_id", "client_id", "trend_direction", "days_since_last_update",
    "impressions_90d", "content_age_days", "word_count", "char_count",
    "search_volume", "competition", "cpc", "content_type", "main_intent",
    "competition_level",
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Required columns are missing from the dataset: {missing}")

# Label is for evaluation only; it is never added to X.
df[TARGET] = df["trend_direction"].eq("down").astype("int8")
print(f"Observed decline-label base rate: {df[TARGET].mean():.3f}")
print(f"Unique clients available for grouped splitting: {df['client_id'].nunique()}")

# Safe snapshot metadata/context features. No IDs, labels, trend fields,
# direct 30-day trend ingredients, current traffic totals, or product flags.
NUMERIC_FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc",
]
CATEGORICAL_FEATURES = [
    "content_type",
    "main_intent",
    "competition_level",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

FORBIDDEN_FEATURES = {
    "content_id", "client_id", TARGET, "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "health_score", "needs_ctr_fix", "is_quick_win",
}
assert not (set(FEATURES) & FORBIDDEN_FEATURES), "A forbidden or label-derived field entered the feature list."
assert set(FEATURES).issubset(df.columns), "At least one model feature is missing."

print("\nModel features:")
print(", ".join(FEATURES))
print("\nLeakage check: no target, trend-derived fields, IDs, current traffic totals, or product flags are model inputs.")


Loaded dataset: 30,000 rows × 44 columns
Data source: anonymized starter CSV; no client names, domains, or URLs are displayed.
Observed decline-label base rate: 0.542
Unique clients available for grouped splitting: 32

Model features:
content_age_days, days_since_last_update, word_count, char_count, search_volume, competition, cpc, content_type, main_intent, competition_level

Leakage check: no target, trend-derived fields, IDs, current traffic totals, or product flags are model inputs.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a **grouped holdout split by `client_id`**. A client's pages stay together, so pages from the same client cannot appear in both training and test data. This is more honest than randomly splitting rows when the goal is to check whether the model works for clients it did not train on.

The split is fixed with random seed 42 for repeatability. The Week-4 baseline is evaluated on these same test rows. Since this starter dataset is a snapshot and its label describes an observed trend, the result should not be described as future forecasting.


In [2]:
# Make one reproducible client-grouped train/test split.

from sklearn.model_selection import GroupShuffleSplit

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
groups = df["client_id"].astype(str)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
train_rows = df.iloc[train_idx].copy()
test_rows = df.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients), "Client leakage: a client appears in both splits."
assert len(X_train) > 0 and len(X_test) > 0
assert y_train.nunique() == 2, "Training split must contain both label classes."

print(f"Training rows: {len(X_train):,} ({len(train_clients)} clients)")
print(f"Test rows:     {len(X_test):,} ({len(test_clients)} clients)")
print(f"Train decline base rate: {y_train.mean():.3f}")
print(f"Test decline base rate:  {y_test.mean():.3f}")
print("Client overlap between train and test: 0")


Training rows: 22,885 (24 clients)
Test rows:     7,115 (8 clients)
Train decline base rate: 0.550
Test decline base rate:  0.517
Client overlap between train and test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The baseline is the Week-4 rule: select pages with at least 180 days since their last update and at least 500 impressions over 90 days, then rank selected pages by impressions. It is applied to the test split only for evaluation.

The model is a Logistic Regression pipeline. Numeric missing values are filled with the training-set median and missingness indicators; categorical missing values are labeled `"unknown"` and categories are one-hot encoded. The pipeline is fitted only on the training split.

The comparison reports **Precision@20 and Precision@50** for both rankings, plus average precision and the test-set base rate. These metrics are calculated on the same test rows. A higher value means more pages in the top of that ranking have the observed decline label; it does not mean that a refresh will fix them.


In [3]:
# Train Logistic Regression and compare it with the Week-4 rule on the same test rows.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import average_precision_score

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)
model_probability = model.predict_proba(X_test)[:, 1]
model_prediction = (model_probability >= 0.5).astype(int)

# Week-4 rule baseline: same thresholds and tie-break ordering as the baseline notebook.
baseline_selected = (
    pd.to_numeric(test_rows["days_since_last_update"], errors="coerce").ge(180).fillna(False)
    & pd.to_numeric(test_rows["impressions_90d"], errors="coerce").ge(500).fillna(False)
)
baseline_score = np.where(
    baseline_selected,
    pd.to_numeric(test_rows["impressions_90d"], errors="coerce").fillna(0).to_numpy(),
    0.0,
)

# Use a stable row order for tie-breaking, mirroring the Week-4 rule's rank fields.
baseline_order_frame = pd.DataFrame({
    "row_position": np.arange(len(test_rows)),
    "score": baseline_score,
    "impressions_90d": pd.to_numeric(test_rows["impressions_90d"], errors="coerce").fillna(0).to_numpy(),
    "days_since_last_update": pd.to_numeric(test_rows["days_since_last_update"], errors="coerce").fillna(0).to_numpy(),
    "content_id": test_rows["content_id"].astype(str).to_numpy(),
})
baseline_order = baseline_order_frame.sort_values(
    ["score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False],
    kind="mergesort",
)["row_position"].to_numpy()

model_order = np.argsort(-model_probability, kind="stable")
y_test_array = y_test.to_numpy()
# Convert the exact baseline ordering into unique scores so average precision
# respects the same tie-break order used by Precision@K.
baseline_rank_score = np.empty(len(test_rows), dtype=float)
baseline_rank_score[baseline_order] = np.arange(len(test_rows), 0, -1, dtype=float)

def precision_at_k(y_true, ranking, k):
    k_eff = min(k, len(ranking))
    return float(np.mean(y_true[ranking[:k_eff]])) if k_eff else float("nan")

K_VALUES = [20, 50]
comparison_rows = []
for name, ranking, scores in [
    ("Week-4 rule baseline", baseline_order, baseline_rank_score),
    ("Logistic Regression", model_order, model_probability),
]:
    row = {
        "method": name,
        "test_base_rate": float(y_test.mean()),
        "average_precision": float(average_precision_score(y_test_array, scores)),
    }
    for k in K_VALUES:
        row[f"precision_at_{k}"] = precision_at_k(y_test_array, ranking, k)
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows)
display(comparison.style.format({
    "test_base_rate": "{:.3f}",
    "average_precision": "{:.3f}",
    "precision_at_20": "{:.3f}",
    "precision_at_50": "{:.3f}",
}))

print("Metric note: Precision@K is the share of the top K ranked test pages with the observed decline label.")
print("Average precision summarizes the full ranking. Higher is better; compare each method on the same test rows.")


,method,test_base_rate,average_precision,precision_at_20,precision_at_50
0,Week-4 rule baseline,0.517,0.504,0.450,0.440
1,Logistic Regression,0.517,0.529,0.600,0.640


Metric note: Precision@K is the share of the top K ranked test pages with the observed decline label.
Average precision summarizes the full ranking. Higher is better; compare each method on the same test rows.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I inspect false positives and false negatives at the default probability threshold of 0.50, and review permutation importance to see which allowed input columns matter most to average precision. A false positive is a page the model flags as declining but whose observed label is not down; a false negative is a declining page the model misses at that threshold.

The top feature list is an interpretation aid, not proof of cause. The three error examples use pseudonymous IDs only. The model may miss effects of seasonality, page intent, measurement gaps, and client-specific context that are not represented by these features.


In [4]:
# Inspect classification errors, estimate feature importance, and save a compact metrics receipt.

from sklearn.inspection import permutation_importance

error_frame = test_rows[["content_id", "client_id"]].copy()
error_frame["actual_decline"] = y_test.to_numpy()
error_frame["predicted_decline"] = model_prediction
error_frame["predicted_probability"] = model_probability
error_frame["error_type"] = np.select(
    [
        (error_frame["actual_decline"] == 0) & (error_frame["predicted_decline"] == 1),
        (error_frame["actual_decline"] == 1) & (error_frame["predicted_decline"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)
error_frame["days_since_last_update"] = test_rows["days_since_last_update"].to_numpy()
error_frame["content_age_days"] = test_rows["content_age_days"].to_numpy()
error_frame["content_type"] = test_rows["content_type"].astype(str).to_numpy()

misclassified = error_frame.loc[error_frame["error_type"] != "correct"].copy()
print(f"Rows misclassified at threshold 0.50: {len(misclassified):,} / {len(test_rows):,}")
if misclassified.empty:
    print("No misclassified rows at threshold 0.50 in this test split.")
else:
    print("Three example errors (pseudonymous IDs only):")
    display(misclassified.assign(
        predicted_probability=misclassified["predicted_probability"].round(3)
    ).head(3))

print("\nMost confident false positives:")
false_positives = error_frame.loc[error_frame["error_type"] == "false_positive"].sort_values(
    "predicted_probability", ascending=False
).head(3)
display(false_positives if not false_positives.empty else pd.DataFrame({"note": ["No false positives at threshold 0.50."]}))

print("\nMost confident false negatives:")
false_negatives = error_frame.loc[error_frame["error_type"] == "false_negative"].sort_values(
    "predicted_probability", ascending=True
).head(3)
display(false_negatives if not false_negatives.empty else pd.DataFrame({"note": ["No false negatives at threshold 0.50."]}))

# Permutation importance on held-out rows: the drop in average precision after shuffling a feature.
importance_result = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance = pd.DataFrame({
    "feature": FEATURES,
    "mean_average_precision_drop": importance_result.importances_mean,
    "std": importance_result.importances_std,
}).sort_values("mean_average_precision_drop", ascending=False)
print("\nPermutation importance (larger positive drop = more useful to this model on this test set):")
display(importance.round(4))

# Baseline and model metrics are recorded without saving page-level predictions.
metrics = {
    "task": "rank pages by observed decline label in the anonymized starter snapshot",
    "target": TARGET,
    "method": "LogisticRegression with median imputation, missingness indicators, and one-hot categorical encoding",
    "split": {
        "type": "GroupShuffleSplit by client_id",
        "test_size": 0.25,
        "random_state": RANDOM_STATE,
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "train_client_count": int(len(train_clients)),
        "test_client_count": int(len(test_clients)),
        "client_overlap": 0,
    },
    "feature_columns": FEATURES,
    "test_base_rate": float(y_test.mean()),
    "comparison": comparison.to_dict(orient="records"),
    "top_permutation_features": [
        {
            "feature": str(row["feature"]),
            "mean_average_precision_drop": float(row["mean_average_precision_drop"]),
            "std": float(row["std"]),
        }
        for _, row in importance.head(5).iterrows()
    ],
    "caveats": [
        "The target is an observed snapshot trend, not a future-period label.",
        "No direct label-derived trend fields, IDs, current traffic totals, or product flags are model features.",
        "The baseline rule uses its original 90-day impressions signal; compare as an operational benchmark, not a causal test.",
        "Permutation importance describes association within this split and does not establish causation.",
    ],
}

outputs_dir = repo_root / "work" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
metrics_path = outputs_dir / "model_metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, allow_nan=False)

# Final automatic checks.
assert train_clients.isdisjoint(test_clients)
assert not (set(FEATURES) & FORBIDDEN_FEATURES)
assert set(FEATURES).issubset(df.columns)
assert metrics["split"]["client_overlap"] == 0
assert metrics_path.is_file()

print(f"\nSaved metrics receipt: {metrics_path}")
print("No page-level ranked CSV is written by this notebook.")
print("\nInterpretation guide:")
print("- Compare Precision@20 and Precision@50 first; the higher value puts more observed declines near the top.")
print("- Check whether Logistic Regression improves on the Week-4 rule. If it does not, keep the simpler rule.")
print("- Treat feature importance as a clue for review, not as a causal explanation.")


Rows misclassified at threshold 0.50: 3,343 / 7,115
Three example errors (pseudonymous IDs only):


,content_id,client_id,actual_decline,predicted_decline,predicted_probability,error_type,days_since_last_update,content_age_days,content_type
1,content_a1fb4e703a9e,client_4e07408562,1,0,0.469,false_negative,25,445,keyword article
13,content_a5a2fbc76336,client_8527a891e2,0,1,0.698,false_positive,103,238,keyword article
23,content_2da6ae9d0882,client_e629fa6598,1,0,0.363,false_negative,20,502,keyword article



Most confident false positives:


,content_id,client_id,actual_decline,predicted_decline,predicted_probability,error_type,days_since_last_update,content_age_days,content_type
10138,content_49001e07f686,client_f369cb89fc,0,1,0.800746,false_positive,20,126,keyword article
8744,content_1ff47abbc82b,client_f369cb89fc,0,1,0.800506,false_positive,20,126,keyword article
22460,content_0f2016e46347,client_f369cb89fc,0,1,0.799959,false_positive,20,126,keyword article



Most confident false negatives:


,content_id,client_id,actual_decline,predicted_decline,predicted_probability,error_type,days_since_last_update,content_age_days,content_type
13922,content_420c7df3a60e,client_e629fa6598,1,0,0.194037,false_negative,20,502,keyword article
13778,content_80c0d41cdc85,client_e629fa6598,1,0,0.194037,false_negative,20,502,keyword article
24089,content_43fb1d5c60f3,client_e629fa6598,1,0,0.195740,false_negative,22,502,keyword article



Permutation importance (larger positive drop = more useful to this model on this test set):


,feature,mean_average_precision_drop,std
0,content_age_days,0.0217,0.0043
2,word_count,0.0035,0.0008
6,cpc,0.0014,0.0011
4,search_volume,0.0014,0.0012
7,content_type,0.0009,0.0002
5,competition,-0.0006,0.0008
9,competition_level,-0.0036,0.0026
8,main_intent,-0.0036,0.0006
3,char_count,-0.0059,0.0052
1,days_since_last_update,-0.0177,0.0011



Saved metrics receipt: /content/flyrank-ml-internship/work/outputs/model_metrics.json
No page-level ranked CSV is written by this notebook.

Interpretation guide:
- Compare Precision@20 and Precision@50 first; the higher value puts more observed declines near the top.
- Check whether Logistic Regression improves on the Week-4 rule. If it does not, keep the simpler rule.
- Treat feature importance as a clue for review, not as a causal explanation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.